In [2]:
# ============================
# CORRECTED BATCH EXTREME EVENT FORECASTING FOR ALL STATIONS
# ============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, roc_curve
from sklearn.model_selection import train_test_split
import json
import warnings
warnings.filterwarnings('ignore')

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------- Configuration ----------
EXTREME_RESULTS_DIR = "results_all_stations_extreme_events"
os.makedirs(EXTREME_RESULTS_DIR, exist_ok=True)

# Define all station paths
STATION_PATHS = {
    "Aljarafe": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Aljarafe Results\results_dmoff_consistent_metrics",
    "Asomadilla": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Asomadilla Results\results_dmoff_consistent_metrics", 
    "Bermejales": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Bermejales Results\results_dmoff_consistent_metrics",
    "RonsadelVAalle": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\RonsadelVAalle Results\results_dmoff_consistent_metrics",
    "Torneo": r"E:\Abroad period research\Time-Series KD + Explainability\Multimodel codes\Torneo Results\results_dmoff_consistent_metrics"
}

print("=== CORRECTED BATCH EXTREME EVENT FORECASTING FOR ALL STATIONS ===")
print(f"Processing {len(STATION_PATHS)} stations...")

# ---------- Extreme Event Forecaster Class ----------
class ExtremeEventForecaster:
    def __init__(self, target_col='O3', threshold_quantile=0.95):
        self.target_col = target_col
        self.threshold_quantile = threshold_quantile
        self.models = {}
        self.threshold = None
        
    def define_extreme_events(self, df):
        """Identify extreme events based on percentile threshold"""
        self.threshold = df[self.target_col].quantile(self.threshold_quantile)
        df = df.copy()
        df['extreme_event'] = (df[self.target_col] >= self.threshold).astype(int)
        return df
    
    def prepare_features_extreme(self, df, base_features):
        """Prepare features for extreme event prediction"""
        df = df.copy()
        
        # Use available base features
        available_features = [f for f in base_features if f in df.columns]
        
        # Add rolling statistics for trend detection
        for window in [6, 12, 24]:  # 6h, 12h, 24h windows
            try:
                df[f'{self.target_col}_rolling_mean_{window}'] = df[self.target_col].rolling(window=window, min_periods=1).mean()
                df[f'{self.target_col}_rolling_std_{window}'] = df[self.target_col].rolling(window=window, min_periods=1).std()
            except Exception:
                continue
        
        # Add rate of change features
        try:
            df[f'{self.target_col}_change_1h'] = df[self.target_col].diff(1)
            df[f'{self.target_col}_change_3h'] = df[self.target_col].diff(3)
        except Exception:
            pass
        
        # Fill NaN values
        df = df.fillna(method='bfill').fillna(method='ffill')
        
        # Get all available features for extreme events
        extreme_features = available_features + [
            col for col in df.columns 
            if 'rolling' in col or 'change' in col
        ]
        
        return df, extreme_features
    
    def train_extreme_classifier(self, train_df, features):
        """Train classifier for extreme event prediction"""
        # Prepare features - ensure they exist
        available_features = [f for f in features if f in train_df.columns]
        
        X = train_df[available_features]
        y = train_df['extreme_event']
        
        # Check class distribution
        class_counts = y.value_counts()
        extreme_ratio = y.mean()
        
        # Split data with stratification
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
        )
        
        # Train classifier
        clf = RandomForestClassifier(
            n_estimators=100, 
            random_state=RANDOM_SEED,
            class_weight='balanced'
        )
        
        clf.fit(X_train, y_train)
        
        # Get probabilities and metrics
        y_pred_proba = clf.predict_proba(X_val)[:, 1]
        auroc = roc_auc_score(y_val, y_pred_proba)
        
        # Find optimal threshold using F1-score
        precision, recall, thresholds = precision_recall_curve(y_val, y_pred_proba)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
        optimal_idx = np.argmax(f1_scores[:-1])
        optimal_threshold = thresholds[optimal_idx]
        
        # Convert to binary predictions
        y_pred_binary = (y_pred_proba >= optimal_threshold).astype(int)
        
        # Get classification report for metrics
        report = classification_report(y_val, y_pred_binary, output_dict=True)
        
        results = {
            'model': clf,
            'auroc': auroc,
            'optimal_threshold': optimal_threshold,
            'features_used': available_features,
            'class_distribution': class_counts.to_dict(),
            'extreme_ratio': extreme_ratio,
            'classification_report': report
        }
        
        return results
    
    def early_warning_analysis(self, df, features, lead_times=[1, 3, 6, 12]):
        """Analyze how early we can predict extreme events"""
        early_warning_results = {}
        
        available_features = [f for f in features if f in df.columns]
        
        for lead_time in lead_times:
            try:
                # Create lead target
                df_lead = df.copy()
                df_lead['extreme_event_lead'] = df_lead['extreme_event'].shift(-lead_time)
                df_lead = df_lead.dropna(subset=['extreme_event_lead'])
                
                if len(df_lead) == 0:
                    continue
                
                # Check if we have enough extreme events
                extreme_ratio = df_lead['extreme_event_lead'].mean()
                if extreme_ratio < 0.01:  # Less than 1% extreme events
                    continue
                
                # Train classifier for this lead time
                X = df_lead[available_features]
                y = df_lead['extreme_event_lead']
                
                clf = RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED, class_weight='balanced')
                clf.fit(X, y)
                
                # Predict probabilities
                y_pred_proba = clf.predict_proba(X)[:, 1]
                auroc = roc_auc_score(y, y_pred_proba)
                
                early_warning_results[lead_time] = {
                    'auroc': auroc,
                    'extreme_ratio': extreme_ratio,
                    'n_samples': len(df_lead)
                }
                
            except Exception:
                continue
        
        return early_warning_results

# ---------- Main Processing Function ----------
def process_station_extreme_events(station_name, station_path):
    """Process extreme event forecasting for a single station"""
    print(f"\n{'='*60}")
    print(f"PROCESSING EXTREME EVENTS: {station_name}")
    print(f"{'='*60}")
    
    # Create station-specific results directory
    station_extreme_dir = os.path.join(EXTREME_RESULTS_DIR, f"Extreme_{station_name}")
    os.makedirs(station_extreme_dir, exist_ok=True)
    
    try:
        # Step 1: Load Data
        print("Step 1: Loading DMOFF artifacts...")
        test_df = pd.read_csv(os.path.join(station_path, "test_predictions_dmoff.csv"))
        print(f"Loaded test data: {len(test_df)} samples")
        
        # Step 2: Define Feature Columns
        print("Step 2: Defining feature columns...")
        tab_features = [col for col in test_df.columns if any(x in col for x in ['_lag', 'hour', 'month', 'weekday', 'CO', 'DD', 'NO2', 'PM10', 'TMP', 'VV'])
                       and col not in ['obs', 'pred_dmoff', 'regime', 'FECHA_HORA', 'error']]
        
        # Add time features that exist
        time_features_to_add = []
        for feature in ['hour', 'weekday', 'month', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']:
            if feature in test_df.columns:
                time_features_to_add.append(feature)
        
        base_features = tab_features + time_features_to_add
        print(f"Base features: {len(base_features)}")
        
        # Step 3: Initialize Extreme Event Forecaster
        print("Step 3: Initializing extreme event forecaster...")
        extreme_forecaster = ExtremeEventForecaster(target_col='obs', threshold_quantile=0.95)
        
        # Step 4: Define Extreme Events
        print("Step 4: Defining extreme events...")
        test_extreme = extreme_forecaster.define_extreme_events(test_df)
        threshold = extreme_forecaster.threshold
        extreme_count = test_extreme['extreme_event'].sum()
        extreme_ratio = test_extreme['extreme_event'].mean()
        
        print(f"  Extreme event threshold: {threshold:.3f}")
        print(f"  Extreme events: {extreme_count}/{len(test_extreme)} ({extreme_ratio:.3%})")
        
        # Step 5: Prepare Enhanced Features
        print("Step 5: Preparing enhanced features...")
        test_extreme, extreme_features = extreme_forecaster.prepare_features_extreme(test_extreme, base_features)
        print(f"  Enhanced features: {len(extreme_features)}")
        
        # Step 6: Train-Test Split for Evaluation
        print("Step 6: Creating train-test split...")
        train_extreme, eval_extreme = train_test_split(
            test_extreme, test_size=0.3, random_state=RANDOM_SEED, stratify=test_extreme['extreme_event']
        )
        print(f"  Training samples: {len(train_extreme)}, Evaluation samples: {len(eval_extreme)}")
        
        # Step 7: Train Extreme Event Classifier
        print("Step 7: Training extreme event classifier...")
        classifier_results = extreme_forecaster.train_extreme_classifier(train_extreme, extreme_features)
        
        auroc = classifier_results['auroc']
        optimal_threshold = classifier_results['optimal_threshold']
        
        print(f"  Validation AUROC: {auroc:.3f}")
        print(f"  Optimal threshold: {optimal_threshold:.3f}")
        
        # Step 8: Evaluate on Test Set
        print("Step 8: Evaluating on evaluation set...")
        best_model = classifier_results['model']
        features_used = classifier_results['features_used']
        
        # Ensure evaluation set has the same features
        available_eval_features = [f for f in features_used if f in eval_extreme.columns]
        
        # Predict on evaluation set
        eval_probs = best_model.predict_proba(eval_extreme[available_eval_features])[:, 1]
        eval_predictions = (eval_probs >= optimal_threshold).astype(int)
        
        # Calculate evaluation metrics
        eval_auroc = roc_auc_score(eval_extreme['extreme_event'], eval_probs)
        eval_report = classification_report(eval_extreme['extreme_event'], eval_predictions, output_dict=True)
        
        print(f"  Evaluation AUROC: {eval_auroc:.3f}")
        
        # CORRECTED: Extract metrics from classification report properly
        if '1' in eval_report:
            precision_val = eval_report['1']['precision']
            recall_val = eval_report['1']['recall']
            f1_val = eval_report['1']['f1-score']
            print(f"  Precision: {precision_val:.3f}, Recall: {recall_val:.3f}, F1-Score: {f1_val:.3f}")
        else:
            precision_val = 0.0
            recall_val = 0.0
            f1_val = 0.0
            print("  No extreme events detected in evaluation set")
        
        # Step 9: Early Warning Analysis
        print("Step 9: Performing early warning analysis...")
        early_warning_results = extreme_forecaster.early_warning_analysis(
            train_extreme, extreme_features, lead_times=[1, 3, 6, 12]
        )
        
        print(f"  Early warning analysis completed for {len(early_warning_results)} lead times")
        
        # Step 10: Create Visualizations
        print("Step 10: Creating visualizations...")
        plt.figure(figsize=(15, 10))
        
        # Plot 1: ROC Curve
        plt.subplot(2, 3, 1)
        fpr, tpr, _ = roc_curve(eval_extreme['extreme_event'], eval_probs)
        plt.plot(fpr, tpr, label=f'ROC (AUC = {eval_auroc:.3f})', linewidth=2)
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Plot 2: Precision-Recall Curve
        plt.subplot(2, 3, 2)
        precision_curve, recall_curve, _ = precision_recall_curve(eval_extreme['extreme_event'], eval_probs)
        plt.plot(recall_curve, precision_curve, linewidth=2)
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curve')
        plt.grid(True, alpha=0.3)
        
        # Plot 3: Early Warning Performance
        plt.subplot(2, 3, 3)
        if early_warning_results:
            lead_times = list(early_warning_results.keys())
            aurocs = [early_warning_results[lt]['auroc'] for lt in lead_times]
            plt.plot(lead_times, aurocs, 'o-', linewidth=2, markersize=8)
            plt.xlabel('Lead Time (hours)')
            plt.ylabel('AUROC')
            plt.title('Early Warning Performance')
            plt.grid(True, alpha=0.3)
        
        # Plot 4: Feature Importance
        plt.subplot(2, 3, 4)
        if hasattr(best_model, 'feature_importances_'):
            feature_imp = pd.DataFrame({
                'feature': features_used,
                'importance': best_model.feature_importances_
            }).sort_values('importance', ascending=True).tail(10)
            
            plt.barh(feature_imp['feature'], feature_imp['importance'])
            plt.title('Top 10 Feature Importance')
            plt.xlabel('Importance')
        
        # Plot 5: Probability Distribution
        plt.subplot(2, 3, 5)
        plt.hist(eval_probs[eval_extreme['extreme_event'] == 0], 
                 alpha=0.7, label='Non-Extreme', bins=20, color='blue', density=True)
        plt.hist(eval_probs[eval_extreme['extreme_event'] == 1], 
                 alpha=0.7, label='Extreme', bins=20, color='red', density=True)
        plt.axvline(x=optimal_threshold, color='black', linestyle='--', 
                    label=f'Threshold ({optimal_threshold:.2f})')
        plt.xlabel('Predicted Probability')
        plt.ylabel('Density')
        plt.title('Probability Distribution')
        plt.legend()
        
        # Plot 6: Confusion Matrix
        plt.subplot(2, 3, 6)
        cm = confusion_matrix(eval_extreme['extreme_event'], eval_predictions)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        
        plt.tight_layout()
        plt.savefig(os.path.join(station_extreme_dir, f"extreme_events_analysis_{station_name}.png"), 
                    dpi=300, bbox_inches='tight')
        plt.close()
        
        # Step 11: Save Station Results
        print("Step 11: Saving station results...")
        
        # Save predictions
        eval_results = eval_extreme.copy()
        eval_results['extreme_probability'] = eval_probs
        eval_results['extreme_prediction'] = eval_predictions
        eval_results.to_csv(os.path.join(station_extreme_dir, f"extreme_event_predictions_{station_name}.csv"), index=False)
        
        # Create station summary - CORRECTED: Use the extracted values, not arrays
        station_summary = {
            'station': station_name,
            'threshold': float(threshold),
            'extreme_count': int(extreme_count),
            'extreme_ratio': float(extreme_ratio),
            'validation_auroc': float(auroc),
            'evaluation_auroc': float(eval_auroc),
            'optimal_threshold': float(optimal_threshold),
            'precision': float(precision_val),
            'recall': float(recall_val),
            'f1_score': float(f1_val),
            'n_features': len(features_used),
            'early_warning_lead_times': len(early_warning_results)
        }
        
        # Add early warning results
        for lead_time, results in early_warning_results.items():
            station_summary[f'early_warning_{lead_time}h_auroc'] = float(results['auroc'])
        
        # Save station summary
        with open(os.path.join(station_extreme_dir, f"extreme_event_summary_{station_name}.json"), "w") as f:
            json.dump(station_summary, f, indent=2)
        
        print(f"✓ Station {station_name} extreme event analysis completed!")
        return station_summary
        
    except Exception as e:
        print(f"❌ Error processing station {station_name}: {e}")
        import traceback
        traceback.print_exc()
        return {
            'station': station_name,
            'error': str(e)
        }

# ---------- Main Execution ----------
def main():
    """Main function to process all stations"""
    print("Starting batch extreme event forecasting...")
    
    all_station_summaries = []
    
    # Process each station
    for station_name, station_path in STATION_PATHS.items():
        if os.path.exists(station_path):
            station_summary = process_station_extreme_events(station_name, station_path)
            all_station_summaries.append(station_summary)
        else:
            print(f"❌ Station path not found: {station_path}")
            all_station_summaries.append({
                'station': station_name,
                'error': f"Path {station_path} does not exist"
            })
    
    # Create Comprehensive Paper-Ready Table
    print(f"\n{'='*80}")
    print("CREATING COMPREHENSIVE EXTREME EVENT RESULTS TABLE")
    print(f"{'='*80}")
    
    # Convert to DataFrame
    summary_df = pd.DataFrame(all_station_summaries)
    
    # Create paper-ready table
    paper_table = []
    
    for station in summary_df.to_dict('records'):
        if 'error' not in station:
            paper_table.append({
                'Station': station['station'],
                'Threshold': f"{station.get('threshold', 0):.1f}",
                'Extreme_Events': f"{station.get('extreme_count', 0)}",
                'Extreme_Ratio': f"{station.get('extreme_ratio', 0):.3f}",
                'AUROC': f"{station.get('evaluation_auroc', 0):.3f}",
                'Precision': f"{station.get('precision', 0):.3f}",
                'Recall': f"{station.get('recall', 0):.3f}",
                'F1_Score': f"{station.get('f1_score', 0):.3f}",
                'Early_Warning_1h': f"{station.get('early_warning_1h_auroc', 'N/A')}",
                'Early_Warning_6h': f"{station.get('early_warning_6h_auroc', 'N/A')}",
            })
        else:
            paper_table.append({
                'Station': station['station'],
                'Threshold': 'Failed',
                'Extreme_Events': 'Failed',
                'Extreme_Ratio': 'Failed',
                'AUROC': 'Failed',
                'Precision': 'Failed',
                'Recall': 'Failed',
                'F1_Score': 'Failed',
                'Early_Warning_1h': 'Failed',
                'Early_Warning_6h': 'Failed',
            })
    
    # Create final paper table
    paper_df = pd.DataFrame(paper_table)
    
    # Save comprehensive results
    paper_df.to_csv(os.path.join(EXTREME_RESULTS_DIR, "PAPER_READY_EXTREME_EVENTS_TABLE.csv"), index=False)
    summary_df.to_csv(os.path.join(EXTREME_RESULTS_DIR, "detailed_extreme_events_summary.csv"), index=False)
    
    # Print beautiful final table
    print("\n" + "="*100)
    print("FINAL EXTREME EVENT FORECASTING RESULTS")
    print("="*100)
    
    # Display formatted table
    display_columns = ['Station', 'Threshold', 'Extreme_Events', 'Extreme_Ratio', 'AUROC', 'Precision', 'Recall', 'F1_Score']
    
    display_df = paper_df[display_columns].copy()
    display_df.columns = ['Station', 'Threshold', 'Extreme Events', 'Extreme Ratio', 'AUROC', 'Precision', 'Recall', 'F1-Score']
    
    print(display_df.to_string(index=False))
    
    # Calculate overall statistics
    successful_stations = [s for s in all_station_summaries if 'error' not in s]
    if successful_stations:
        avg_auroc = np.mean([s.get('evaluation_auroc', 0) for s in successful_stations])
        avg_precision = np.mean([s.get('precision', 0) for s in successful_stations])
        avg_recall = np.mean([s.get('recall', 0) for s in successful_stations])
        avg_f1 = np.mean([s.get('f1_score', 0) for s in successful_stations])
        
        print(f"\n{'='*80}")
        print("OVERALL SUMMARY STATISTICS (Successful Stations Only)")
        print(f"{'='*80}")
        print(f"Number of successful stations: {len(successful_stations)}/{len(STATION_PATHS)}")
        print(f"Average AUROC: {avg_auroc:.3f}")
        print(f"Average Precision: {avg_precision:.3f}")
        print(f"Average Recall: {avg_recall:.3f}")
        print(f"Average F1-Score: {avg_f1:.3f}")
        
        # Find best performing station
        best_station = max(successful_stations, key=lambda x: x.get('f1_score', 0))
        print(f"Best performing station: {best_station['station']} (F1-Score: {best_station['f1_score']:.3f})")
    
    print(f"\n📁 All results saved to: {EXTREME_RESULTS_DIR}")
    print("✅ BATCH EXTREME EVENT FORECASTING COMPLETED SUCCESSFULLY!")
    print("="*100)

# ---------- Run Main Function ----------
if __name__ == "__main__":
    main()

=== CORRECTED BATCH EXTREME EVENT FORECASTING FOR ALL STATIONS ===
Processing 5 stations...
Starting batch extreme event forecasting...

PROCESSING EXTREME EVENTS: Aljarafe
Step 1: Loading DMOFF artifacts...
Loaded test data: 8511 samples
Step 2: Defining feature columns...
Base features: 43
Step 3: Initializing extreme event forecaster...
Step 4: Defining extreme events...
  Extreme event threshold: 118.333
  Extreme events: 427/8511 (5.017%)
Step 5: Preparing enhanced features...
  Enhanced features: 51
Step 6: Creating train-test split...
  Training samples: 5957, Evaluation samples: 2554
Step 7: Training extreme event classifier...
  Validation AUROC: 0.997
  Optimal threshold: 0.330
Step 8: Evaluating on evaluation set...
  Evaluation AUROC: 0.995
  Precision: 0.812, Recall: 0.875, F1-Score: 0.842
Step 9: Performing early warning analysis...
  Early warning analysis completed for 4 lead times
Step 10: Creating visualizations...
Step 11: Saving station results...
✓ Station Aljarafe